# CustomerPulse AI

# Phase 2: Data Cleaning & Preprocessing

## Objective

The objective of this notebook is to clean and preprocess the raw Olist E-commerce datasets before feature engineering and analysis.

## Tasks

- Load raw datasets
- Handle missing values
- Fix incorrect data types
- Remove duplicate records
- Validate data quality
- Save cleaned datasets

## Output

A clean and reliable dataset ready for feature engineering and business analysis.

In [2]:
# ==========================================================
# CustomerPulse AI
# Phase 2 - Data Cleaning & Preprocessing
# ==========================================================

import pandas as pd
import numpy as np

import os

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [3]:
# ==========================================================
# Dataset Paths
# ==========================================================

RAW_PATH = "../1 data/01_raw_data/"
PROCESSED_PATH = "../1 data/02_processed_data/"

In [4]:
# ==========================================================
# Load Raw Datasets
# ==========================================================

customers = pd.read_csv(RAW_PATH + "olist_customers_dataset.csv")

orders = pd.read_csv(RAW_PATH + "olist_orders_dataset.csv")

order_items = pd.read_csv(RAW_PATH + "olist_order_items_dataset.csv")

payments = pd.read_csv(RAW_PATH + "olist_order_payments_dataset.csv")

reviews = pd.read_csv(RAW_PATH + "olist_order_reviews_dataset.csv")

products = pd.read_csv(RAW_PATH + "olist_products_dataset.csv")

sellers = pd.read_csv(RAW_PATH + "olist_sellers_dataset.csv")

In [5]:
# ==========================================================
# Dataset Dictionary
# ==========================================================

datasets = {

    "Customers": customers,
    "Orders": orders,
    "Order_Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers

}

In [6]:
# ==========================================================
# Dataset Shape Before Cleaning
# ==========================================================

summary = []

for name, df in datasets.items():

    summary.append({

        "Dataset": name,

        "Rows": df.shape[0],

        "Columns": df.shape[1]

    })

pd.DataFrame(summary)

,Dataset,Rows,Columns
0,Customers,99441,5
1,Orders,99441,8
2,Order_Items,112650,7
3,Payments,103886,5
4,Reviews,99224,7
5,Products,32951,9
6,Sellers,3095,4


In [7]:
# ==========================================================
# Missing Value Report
# ==========================================================

missing_report = []

for name, df in datasets.items():

    for column in df.columns:

        missing = df[column].isnull().sum()

        if missing > 0:

            missing_report.append({

                "Dataset": name,

                "Column": column,

                "Missing Values": missing,

                "Missing %": round(
                    missing / len(df) * 100,
                    2
                )

            })

missing_report = pd.DataFrame(missing_report)

missing_report.sort_values(
    by="Missing %",
    ascending=False
).reset_index(drop=True)

,Dataset,Column,Missing Values,Missing %
0,Reviews,review_comment_title,87656,88.34
1,Reviews,review_comment_message,58247,58.70
2,Orders,order_delivered_customer_date,2965,2.98
3,Products,product_name_lenght,610,1.85
4,Products,product_category_name,610,1.85
5,Products,product_description_lenght,610,1.85
6,Products,product_photos_qty,610,1.85
7,Orders,order_delivered_carrier_date,1783,1.79
8,Orders,order_approved_at,160,0.16
9,Products,product_weight_g,2,0.01


In [8]:
# ==========================================================
# Missing Value Summary
# ==========================================================

missing_report.groupby("Dataset")[
    "Missing Values"
].sum().reset_index()

,Dataset,Missing Values
0,Orders,4908
1,Products,2448
2,Reviews,145903


# Missing Value Strategy

### Customers
- No missing values

### Orders
- Datetime columns contain missing values due to cancelled or unavailable orders.
- Missing values will be retained where they represent real business events.

### Reviews
- Missing review comments indicate customers submitted ratings only.
- Missing text fields will not be filled.

### Products
- Product attributes contain missing values.
- Appropriate treatment will be applied based on business importance.

### Payments
- No missing values

### Sellers
- No missing values


================ DATA CLEANING PLAN ================

1. Convert date columns to datetime.

2. Validate duplicate records.

3. Handle missing values.

4. Validate numerical columns.

5. Validate categorical columns.

6. Perform consistency checks.

7. Save cleaned datasets.

====================================================


In [10]:
# ==========================================================
# Current Data Types
# ==========================================================

for name, df in datasets.items():

    print("="*80)
    print(name)
    print("="*80)

    display(df.dtypes.to_frame(name="Current Data Type"))

Customers


,Current Data Type
customer_id,str
customer_unique_id,str
customer_zip_code_prefix,int64
customer_city,str
customer_state,str


Orders


,Current Data Type
order_id,str
customer_id,str
order_status,str
order_purchase_timestamp,str
order_approved_at,str
order_delivered_carrier_date,str
order_delivered_customer_date,str
order_estimated_delivery_date,str


Order_Items


...

,Current Data Type
review_id,str
order_id,str
review_score,int64
review_comment_title,str
review_comment_message,str
review_creation_date,str
review_answer_timestamp,str


Products


,Current Data Type
product_id,str
product_category_name,str
product_name_lenght,float64
product_description_lenght,float64
product_photos_qty,float64
product_weight_g,float64
product_length_cm,float64
product_height_cm,float64
product_width_cm,float64


Sellers


,Current Data Type
seller_id,str
seller_zip_code_prefix,int64
seller_city,str
seller_state,str


In [11]:
# ==========================================================
# Date Columns to Convert
# ==========================================================

datetime_columns = {

    "Orders": [

        "order_purchase_timestamp",

        "order_approved_at",

        "order_delivered_carrier_date",

        "order_delivered_customer_date",

        "order_estimated_delivery_date"

    ],

    "Reviews": [

        "review_creation_date",

        "review_answer_timestamp"

    ]

}

datetime_columns

{'Orders': ['order_purchase_timestamp',
  'order_approved_at',
  'order_delivered_carrier_date',
  'order_delivered_customer_date',
  'order_estimated_delivery_date'],
 'Reviews': ['review_creation_date', 'review_answer_timestamp']}

In [12]:
# ==========================================================
# Convert Orders Date Columns
# ==========================================================

orders_datetime = [

    "order_purchase_timestamp",

    "order_approved_at",

    "order_delivered_carrier_date",

    "order_delivered_customer_date",

    "order_estimated_delivery_date"

]

for column in orders_datetime:

    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

print("Orders datetime conversion completed.")

Orders datetime conversion completed.


In [13]:
# ==========================================================
# Convert Reviews Date Columns
# ==========================================================

reviews_datetime = [

    "review_creation_date",

    "review_answer_timestamp"

]

for column in reviews_datetime:

    reviews[column] = pd.to_datetime(
        reviews[column],
        errors="coerce"
    )

print("Reviews datetime conversion completed.")

Reviews datetime conversion completed.


In [14]:
# ==========================================================
# Verify Updated Data Types
# ==========================================================

print("Orders\n")
display(orders.dtypes)

print("\nReviews\n")
display(reviews.dtypes)

Orders



order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


Reviews



review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [15]:
# ==========================================================
# Duplicate Validation
# ==========================================================

duplicate_report = []

for name, df in datasets.items():

    duplicate_report.append({

        "Dataset": name,

        "Duplicate Rows": df.duplicated().sum()

    })

duplicate_report = pd.DataFrame(duplicate_report)

duplicate_report

,Dataset,Duplicate Rows
0,Customers,0
1,Orders,0
2,Order_Items,0
3,Payments,0
4,Reviews,0
5,Products,0
6,Sellers,0


In [36]:
# ==========================================================
# Dataset Shapes After Duplicate Removal
# ==========================================================

datasets = {

    "Customers": customers,

    "Orders": orders,

    "Order_Items": order_items,

    "Payments": payments,

    "Reviews": reviews,

    "Products": products,

    "Sellers": sellers

}

summary = []

for name, df in datasets.items():

    summary.append({

        "Dataset": name,

        "Rows": df.shape[0],

        "Columns": df.shape[1]

    })

pd.DataFrame(summary)

,Dataset,Rows,Columns
0,Customers,99441,5
1,Orders,99441,8
2,Order_Items,112650,7
3,Payments,103886,5
4,Reviews,99224,9
5,Products,32951,9
6,Sellers,3095,4


# Business Validation

Not every missing value represents poor data quality.

### Orders
- Missing delivery dates are expected for cancelled or unavailable orders.

### Reviews
- Customers are not required to write review comments.

### Products
- Some product attributes are unavailable due to incomplete catalog information.

Therefore, missing values will be handled based on business logic rather than simply deleting records.

In [ ]:
# ==========================================================
# Customers Dataset Cleaning Missing values
# ==========================================================

print("Missing Values")
display(customers.isnull().sum())

print("\nDuplicate Rows :", customers.duplicated().sum())

Missing Values


customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


Duplicate Rows : 0


In [20]:
# ==========================================================
# Orders Dataset Missing Values
# ==========================================================

orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [21]:
# ==========================================================
# Orders Business Validation
# ==========================================================

orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [39]:
reviews = pd.read_csv(RAW_PATH + "olist_order_reviews_dataset.csv")
reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"]
)

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"]
)
reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [24]:
# ==========================================================
# Products Missing Values
# ==========================================================

products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [25]:
products["product_category_name"] = products[
    "product_category_name"
].fillna("Unknown")

In [31]:
numeric_columns = [

    "product_name_lenght",

    "product_description_lenght",

    "product_photos_qty",

    "product_weight_g",

    "product_length_cm",

    "product_height_cm",

    "product_width_cm"

]

products[numeric_columns].describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


In [32]:
for column in numeric_columns:

    products[column] = products[column].fillna(
        products[column].median()
    )

In [28]:
print("Payments")

display(payments.isnull().sum())

print()

print("Sellers")

display(sellers.isnull().sum())

Payments


order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64


Sellers


seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [33]:
# cleaning validation
cleaning_report = {

    "Customers": customers.isnull().sum().sum(),

    "Orders": orders.isnull().sum().sum(),

    "Payments": payments.isnull().sum().sum(),

    "Reviews": reviews.isnull().sum().sum(),

    "Products": products.isnull().sum().sum(),

    "Sellers": sellers.isnull().sum().sum()

}

pd.DataFrame(

    cleaning_report.items(),

    columns=["Dataset","Remaining Missing Values"]

)

,Dataset,Remaining Missing Values
0,Customers,0
1,Orders,4908
2,Payments,0
3,Reviews,0
4,Products,0
5,Sellers,0


In [34]:
products.isnull().mean() * 100

product_id                    0.0
product_category_name         0.0
product_name_lenght           0.0
product_description_lenght    0.0
product_photos_qty            0.0
product_weight_g              0.0
product_length_cm             0.0
product_height_cm             0.0
product_width_cm              0.0
dtype: float64

In [42]:
for name, df in datasets.items():
    print(name, df.duplicated().sum())

Customers 0
Orders 0
Order_Items 0
Payments 0
Reviews 0
Products 0
Sellers 0


In [48]:
# Reload Reviews
reviews = pd.read_csv(RAW_PATH + "olist_order_reviews_dataset.csv")

# Datetime conversion
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

# Update dictionary
datasets["Reviews"] = reviews

In [49]:
reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [50]:
validation = []

for name, df in datasets.items():

    validation.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing": df.isnull().sum().sum(),
        "Duplicates": df.duplicated().sum()
    })

validation = pd.DataFrame(validation)

validation

,Dataset,Rows,Columns,Missing,Duplicates
0,Customers,99441,5,0,0
1,Orders,99441,8,4908,0
2,Order_Items,112650,7,0,0
3,Payments,103886,5,0,0
4,Reviews,99224,7,145903,0
5,Products,32951,9,0,0
6,Sellers,3095,4,0,0


In [51]:
orders.info()

reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB
<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------         

In [52]:
# ==========================================================
# Final Data Quality Report
# ==========================================================

quality_report = pd.DataFrame({

    "Dataset":[
        "Customers",
        "Orders",
        "Order_Items",
        "Payments",
        "Reviews",
        "Products",
        "Sellers"
    ],

    "Status":[
        "Clean",
        "Business Missing",
        "Clean",
        "Clean",
        "Business Missing",
        "Clean",
        "Clean"
    ],

    "Reason":[

        "No issues detected.",

        "Missing delivery dates represent cancelled/unavailable orders.",

        "No issues detected.",

        "No issues detected.",

        "Review title and comments are optional fields.",

        "Missing product attributes have been handled appropriately.",

        "No issues detected."

    ]

})

quality_report

,Dataset,Status,Reason
0,Customers,Clean,No issues detected.
1,Orders,Business Missing,Missing delivery dates represent cancelled/una...
2,Order_Items,Clean,No issues detected.
3,Payments,Clean,No issues detected.
4,Reviews,Business Missing,Review title and comments are optional fields.
5,Products,Clean,Missing product attributes have been handled a...
6,Sellers,Clean,No issues detected.


In [54]:
import os

os.listdir("../1 data")

['01_raw_data', '02_processed data']

In [55]:
PROCESSED_PATH = "../1 data/02_processed data/"

In [ ]:
# ==========================================================
# Save Cleaned Datasets
# ==========================================================

customers.to_csv(PROCESSED_PATH + "customers_clean.csv", index=False)

orders.to_csv(PROCESSED_PATH + "orders_clean.csv", index=False)

order_items.to_csv(PROCESSED_PATH + "order_items_clean.csv", index=False)

payments.to_csv(PROCESSED_PATH + "payments_clean.csv", index=False)

reviews.to_csv(PROCESSED_PATH + "reviews_clean.csv", index=False)

products.to_csv(PROCESSED_PATH + "products_clean.csv", index=False)

sellers.to_csv(PROCESSED_PATH + "sellers_clean.csv", index=False)

print("=" * 60)
print(" All cleaned datasets saved successfully.")
print(f" Saved to: {PROCESSED_PATH}")
print("=" * 60)

✅ All cleaned datasets saved successfully.
📂 Saved to: ../1 data/02_processed data/
